# LizyML Tutorial: Probability Calibration

This notebook demonstrates **probability calibration** for binary classification in LizyML.

**Why calibration matters:**  
A model may rank predictions well (high AUC) but output poorly calibrated probabilities.
For example, if every sample with true probability 0.3 is predicted as 0.6, the model is
overconfident. Calibration corrects the probability scale so that *predicted probability ≈
observed frequency* — critical for decision thresholds, cost-sensitive scoring, and risk models.

**Calibration methods available in LizyML:**
- `platt` — Logistic regression on OOF predictions (fast, assumes sigmoid shape)
- `isotonic` — Isotonic regression (flexible, needs more data)
- `beta` — Beta calibration (requires `pip install 'lizyml[calibration]'`)

**Contents:**
1. Setup
2. Data
3. Config (Platt calibration)
4. Model Fit
5. Metrics: Raw vs Calibrated
6. Calibration Plot
7. Probability Histogram
8. Compare Calibration Methods

## 1. Setup

In [ ]:
from __future__ import annotations

import pandas as pd
from sklearn.datasets import make_classification

from lizyml import Model

## 2. Data

Synthetic binary classification dataset with 2,000 samples and 8 features.
A larger dataset improves calibration stability, especially for isotonic regression.

In [ ]:
X, y = make_classification(
    n_samples=2000,
    n_features=8,
    n_informative=5,
    n_redundant=2,
    random_state=42,
)

feature_names = [f"feature_{i:02d}" for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feature_names)
df["target"] = y

print(f"Shape: {df.shape}")
print(f"Positive rate: {df['target'].mean():.1%}")
df.head()

## 3. Config

Enable calibration by adding a `calibration` block.
LizyML performs calibration cross-fitting using the same outer CV splits as training,
which prevents leakage — the calibrator never sees data used to train the fold it calibrates.

In [ ]:
config = {
    "config_version": 1,
    "task": "binary",
    "data": {
        "target": "target",
    },
    "model": {
        "name": "lgbm",
        "params": {
            "objective": "binary",
            "n_estimators": 500,
            "learning_rate": 0.05,
            "max_depth": 5,
            "feature_fraction": 0.8,
            "bagging_fraction": 0.8,
            "bagging_freq": 5,
        },
    },
    "split": {
        "method": "stratified_kfold",
        "n_splits": 5,
    },
    "training": {
        "seed": 42,
    },
    "evaluation": {
        "metrics": ["logloss", "auc", "brier", "ece"],
    },
    "calibration": {
        "method": "platt",  # Logistic regression calibration
    },
}

## 4. Model Fit

Calibration is applied automatically after CV training.
The `fit()` call produces both raw and calibrated OOF predictions.

In [ ]:
model = Model(config)
model.fit(data=df)
print("Fit complete.")

## 5. Metrics: Raw vs Calibrated

The evaluate table shows metrics computed on OOF predictions for both
the raw model output and the calibrated probabilities.

**Key metrics to compare:**
- `logloss` — lower is better; sensitive to probability quality
- `brier` — lower is better; mean squared error of probabilities
- `ece` — lower is better; expected calibration error (binned reliability gap)
- `auc` — should be unchanged (calibration preserves ranking)

In [ ]:
model.evaluate_table().round(4)

In [ ]:
# Direct comparison: raw vs calibrated OOF metrics
result = model.evaluate()
raw_oof = result["raw"]["oof"]
cal_oof = result["calibrated"]["oof"]

metrics = ["logloss", "brier", "ece", "auc"]
comparison = pd.DataFrame(
    {
        "raw": {m: raw_oof[m] for m in metrics},
        "calibrated": {m: cal_oof[m] for m in metrics},
    }
)
comparison["delta"] = comparison["calibrated"] - comparison["raw"]
comparison.round(4)

## 6. Calibration Plot (Reliability Diagram)

The reliability diagram plots mean predicted probability vs observed frequency
in equal-width bins. A perfectly calibrated model lies on the diagonal.
Curves above the diagonal indicate underconfidence; below = overconfidence.

In [ ]:
model.calibration_plot().show()

## 7. Probability Histogram

Overlaid histograms of raw and calibrated OOF probabilities, split by class.
Good calibration is visible as the positive class (y=1) concentrating near 1.0
and the negative class (y=0) near 0.0.

In [ ]:
model.probability_histogram_plot().show()

## 8. Compare Calibration Methods

To compare methods, re-run with a different `calibration.method` value.
The config is the only change needed — no other code changes required.

| Method | Description | Notes |
|--------|-------------|-------|
| `platt` | Logistic regression | Fast, works well with enough data |
| `isotonic` | Isotonic regression | More flexible, needs 1000+ samples |
| `beta` | Beta calibration | Requires `pip install 'lizyml[calibration]'` |

In [ ]:
# Compare: isotonic calibration
config_isotonic = {**config, "calibration": {"method": "isotonic"}}

model_isotonic = Model(config_isotonic)
model_isotonic.fit(data=df)

result_iso = model_isotonic.evaluate()
cal_iso = result_iso["calibrated"]["oof"]

print("Isotonic calibration metrics (OOF):")
pd.Series({m: cal_iso[m] for m in metrics}, name="isotonic").round(4).to_frame()

In [ ]:
# Side-by-side reliability diagrams
print("Platt calibration:")
model.calibration_plot().show()

print("Isotonic calibration:")
model_isotonic.calibration_plot().show()

In [ ]:
# Summary: all methods side by side
# (beta calibration requires: pip install 'lizyml[calibration]')
result_platt = model.evaluate()

summary = pd.DataFrame(
    {
        "platt": {m: result_platt["calibrated"]["oof"][m] for m in metrics},
        "isotonic": {m: result_iso[m] for m in metrics},
    }
)
print("Calibration method comparison (calibrated OOF):")
summary.round(4)